# WM-811K 선택 모델 다중 시드 재현성 검증

선택이 끝난 `ce_sqrt_balanced` CNN을 동일한 lot 비중복 분할에서 시드 17, 42, 2026으로 다시 학습합니다. 바꾸는 것은 난수 시드뿐입니다. validation macro-F1의 평균과 표본 표준편차를 주 결과로 보고하며, test는 모델 선택에 사용하지 않고 고정된 평가 자료로만 기록합니다. 가장 좋은 시드를 새 모델로 선택하지 않으며 현재 배포 중인 시드 42 모델도 변경하지 않습니다.

T4 GPU 기준으로 기존 1회가 약 7분이었으므로 3회는 대략 20~30분 걸릴 수 있습니다. 중간 결과는 Google Drive에 남아 같은 셀을 다시 실행하면 완료된 시드는 건너뜁니다.

In [ ]:
!pip install -q pandas==2.2.3 scikit-learn matplotlib

from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, subprocess, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 바꾼 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
LOCAL_DATA_DIR = Path('/content/wm811k_processed')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in ('wafer_maps_64.npy', 'labels.npy'):
    source = DRIVE_DATA_DIR / name
    assert source.is_file(), f'전처리 출력 파일이 없습니다: {source}'
    destination = LOCAL_DATA_DIR / name
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        print('로컬 SSD로 복사:', name)
        shutil.copy2(source, destination)

MAPS = LOCAL_DATA_DIR / 'wafer_maps_64.npy'
LABELS = LOCAL_DATA_DIR / 'labels.npy'
maps = np.load(MAPS, mmap_mode='r')
labels = np.load(LABELS, mmap_mode='r')
assert maps.shape == (172950, 64, 64) and labels.shape == (172950,)
print('학습 데이터 준비 완료:', maps.shape, labels.shape)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


## 학습 코드와 고정 분할 업로드

파일 선택 창이 **두 번** 열립니다. 첫 번째 창에서는 로컬 프로젝트의 `코드/wm811k/train_wm811k_cnn.py`, 두 번째 창에서는 `결과물/wm811k/split_results/split_assignments.csv`를 각각 선택하세요. 브라우저가 같은 이름 뒤에 `(1)`을 붙여도 자동으로 인식합니다.

In [ ]:
def uploaded_bytes(uploaded, stem, suffix):
    candidates = [
        data for name, data in uploaded.items()
        if Path(name).stem.startswith(stem) and name.lower().endswith(suffix)
    ]
    assert len(candidates) == 1, f'{stem}{suffix} 파일을 하나만 선택하세요: {list(uploaded)}'
    return candidates[0]

print('1/2: train_wm811k_cnn.py를 선택하세요.')
script_upload = files.upload()
print('2/2: split_assignments.csv를 선택하세요.')
split_upload = files.upload()

TRAIN_SCRIPT = Path('/content/train_wm811k_cnn.py')
ASSIGNMENTS = Path('/content/split_assignments.csv')
TRAIN_SCRIPT.write_bytes(uploaded_bytes(script_upload, 'train_wm811k_cnn', '.py'))
ASSIGNMENTS.write_bytes(uploaded_bytes(split_upload, 'split_assignments', '.csv'))

EXPECTED_SPLIT_SHA256 = '6c6994db76fbeebcc8e3b64d9791ca5a9a0cf2dfb861f68316f3a924867139ce'
actual_split_sha256 = sha256_file(ASSIGNMENTS)
assert actual_split_sha256 == EXPECTED_SPLIT_SHA256, (
    '프로젝트에서 확정한 split_assignments.csv와 SHA-256이 다릅니다: ' + actual_split_sha256
)
assignments = pd.read_csv(ASSIGNMENTS)
required_columns = {'array_index', 'label_id', 'lot_name', 'split'}
assert required_columns.issubset(assignments.columns)
assert len(assignments) == 172950
assert assignments['array_index'].nunique() == 172950
assert set(assignments['split']) == {'train', 'validation', 'test'}
lot_sets = {
    split: set(assignments.loc[assignments['split'] == split, 'lot_name'].dropna().astype(str))
    for split in ('train', 'validation', 'test')
}
assert not (lot_sets['train'] & lot_sets['validation'])
assert not (lot_sets['train'] & lot_sets['test'])
assert not (lot_sets['validation'] & lot_sets['test'])
print('업로드 및 고정 분할 검증 완료')
print(assignments['split'].value_counts().to_dict())
print('split SHA-256:', actual_split_sha256)


## T4 GPU 다중 시드 학습

아래 셀은 세 시드를 순서대로 학습합니다. 각 실행은 validation macro-F1로 조기 종료하며, test 결과는 그 뒤에 한 번 계산됩니다. 완료된 시드의 설정이 정확히 일치하면 재실행 시 건너뜁니다.

In [ ]:
SEEDS = [17, 42, 2026]
RUN_ROOT = Path('/content/drive/MyDrive/AI_project/WM811K_training/multiseed_reproducibility')
RUN_ROOT.mkdir(parents=True, exist_ok=True)

protocol = {
    'schema_version': 1,
    'experiment_type': 'wm811k_multiseed_reproducibility',
    'purpose': 'stability_evidence_not_model_selection',
    'strategy_id': 'ce_sqrt_balanced',
    'loss': 'cross_entropy',
    'sampling': 'sqrt_balanced',
    'sampling_max_multiplier': 8.0,
    'seeds': SEEDS,
    'epochs': 20,
    'batch_size': 256,
    'patience': 4,
    'minimum_delta': 1e-4,
    'fixed_split': True,
    'split_assignments_sha256': actual_split_sha256,
    'selection_split': 'validation',
    'primary_metric': 'validation_macro_f1',
    'test_used_for_selection': False,
    'deployment_model_changed': False,
    'existing_deployment_seed': 42,
}
(RUN_ROOT / 'protocol.json').write_text(
    json.dumps(protocol, ensure_ascii=False, indent=2), encoding='utf-8'
)

def completed_with_fixed_contract(seed_dir, seed):
    summary_path = seed_dir / 'run_summary.json'
    if not summary_path.is_file():
        return False
    try:
        saved = json.loads(summary_path.read_text(encoding='utf-8'))
        return (
            saved.get('seed') == seed
            and saved.get('loss') == 'cross_entropy'
            and saved.get('sampling') == 'sqrt_balanced'
            and float(saved.get('sampling_max_multiplier')) == 8.0
            and (seed_dir / 'best_model.pt').is_file()
        )
    except (OSError, ValueError, TypeError, json.JSONDecodeError):
        return False

for position, seed in enumerate(SEEDS, start=1):
    seed_dir = RUN_ROOT / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    if completed_with_fixed_contract(seed_dir, seed):
        print(f'[{position}/{len(SEEDS)}] seed {seed}: 완료 결과 재사용')
        continue
    print(f'[{position}/{len(SEEDS)}] seed {seed}: 학습 시작')
    command = [
        sys.executable, '-u', str(TRAIN_SCRIPT),
        '--maps', str(MAPS),
        '--labels', str(LABELS),
        '--assignments', str(ASSIGNMENTS),
        '--output-dir', str(seed_dir),
        '--epochs', '20',
        '--batch-size', '256',
        '--num-workers', '2',
        '--loss', 'cross_entropy',
        '--sampling', 'sqrt_balanced',
        '--sampling-max-multiplier', '8',
        '--patience', '4',
        '--minimum-delta', '0.0001',
        '--seed', str(seed),
    ]
    subprocess.run(command, check=True)
    print(f'[{position}/{len(SEEDS)}] seed {seed}: 완료')
print('세 시드 학습 완료:', RUN_ROOT)


## 평균·표준편차 집계 및 전달용 ZIP 다운로드

표준편차는 3회 실행의 표본 표준편차(`ddof=1`)입니다. Drive에는 전체 체크포인트와 예측이 남지만, 전달용 ZIP에는 검증에 필요한 경량 요약만 포함합니다.

In [ ]:
rows = []
for seed in SEEDS:
    seed_dir = RUN_ROOT / f'seed_{seed}'
    run = json.loads((seed_dir / 'run_summary.json').read_text(encoding='utf-8'))
    assert run['seed'] == seed
    assert run['loss'] == 'cross_entropy' and run['sampling'] == 'sqrt_balanced'
    rows.append({
        'seed': seed,
        'best_epoch': int(run['best_epoch']),
        'epochs_ran': int(run['epochs_ran']),
        'elapsed_seconds': float(run['elapsed_seconds']),
        'validation_accuracy': float(run['validation_metrics']['accuracy']),
        'validation_balanced_accuracy': float(run['validation_metrics']['balanced_accuracy']),
        'validation_macro_f1': float(run['validation_metrics']['macro_f1']),
        'test_accuracy': float(run['test_metrics']['accuracy']),
        'test_balanced_accuracy': float(run['test_metrics']['balanced_accuracy']),
        'test_macro_f1': float(run['test_metrics']['macro_f1']),
        'checkpoint_sha256': sha256_file(seed_dir / 'best_model.pt'),
    })
seed_metrics = pd.DataFrame(rows).sort_values('seed').reset_index(drop=True)
metric_names = [
    'validation_accuracy', 'validation_balanced_accuracy', 'validation_macro_f1',
    'test_accuracy', 'test_balanced_accuracy', 'test_macro_f1',
]
metric_records = []
metric_payload = {}
for metric in metric_names:
    values = seed_metrics[metric].to_numpy(dtype=float)
    stats = {
        'mean': float(np.mean(values)),
        'std': float(np.std(values, ddof=1)),
        'min': float(np.min(values)),
        'max': float(np.max(values)),
    }
    metric_payload[metric] = stats
    metric_records.append({'metric': metric, **stats})
metric_summary = pd.DataFrame(metric_records)

summary = {
    **protocol,
    'seed_count': len(SEEDS),
    'std_ddof': 1,
    'metrics': metric_payload,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
seed_metrics.to_csv(RUN_ROOT / 'seed_metrics.csv', index=False)
metric_summary.to_csv(RUN_ROOT / 'metric_summary.csv', index=False)
(RUN_ROOT / 'reproducibility_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)

fig, ax = plt.subplots(figsize=(8, 4.8))
x = np.arange(len(SEEDS))
width = 0.34
ax.bar(x - width / 2, seed_metrics['validation_macro_f1'], width, label='validation macro-F1')
ax.bar(x + width / 2, seed_metrics['test_macro_f1'], width, label='fixed test macro-F1')
ax.axhline(seed_metrics['validation_macro_f1'].mean(), color='navy', linestyle='--', linewidth=1.2, label='validation mean')
ax.set_xticks(x, [str(seed) for seed in SEEDS])
ax.set_xlabel('random seed')
ax.set_ylabel('score')
ax.set_ylim(0.0, 1.0)
ax.set_title('WM-811K selected CNN: multi-seed reproducibility')
ax.legend(loc='lower right')
fig.tight_layout()
plot_path = RUN_ROOT / 'reproducibility_metrics.png'
fig.savefig(plot_path, dpi=160)
plt.show()

BUNDLE_DIR = Path('/content/wm811k_multiseed_bundle')
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir(parents=True)
for name in ('protocol.json', 'seed_metrics.csv', 'metric_summary.csv', 'reproducibility_summary.json', 'reproducibility_metrics.png'):
    shutil.copy2(RUN_ROOT / name, BUNDLE_DIR / name)
for seed in SEEDS:
    source_dir = RUN_ROOT / f'seed_{seed}'
    destination_dir = BUNDLE_DIR / 'seeds' / f'seed_{seed}'
    destination_dir.mkdir(parents=True, exist_ok=True)
    for name in ('run_summary.json', 'validation_classification_report.csv', 'test_classification_report.csv'):
        shutil.copy2(source_dir / name, destination_dir / name)

ZIP_BASE = Path('/content/wm811k_multiseed_reproducibility_results')
zip_path = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=BUNDLE_DIR))
drive_zip = RUN_ROOT.parent / zip_path.name
shutil.copy2(zip_path, drive_zip)
print(seed_metrics.to_string(index=False))
print('validation macro-F1 평균:', f"{metric_payload['validation_macro_f1']['mean']:.6f}")
print('validation macro-F1 표본 표준편차:', f"{metric_payload['validation_macro_f1']['std']:.6f}")
print('전달용 ZIP:', zip_path, f'{zip_path.stat().st_size / 1024:.1f} KB')
print('Drive 백업:', drive_zip)
files.download(str(zip_path))
